# Protecting Routes in Express.js

To protect specific routes in [Express.js](https://expressjs.com/), use custom middleware functions that validate a user's identity before granting access to the route handler.

Middleware sits between the incoming request and your route handler. It can inspect the request, reject it, enrich it, or pass it along by calling `next()`. Authentication is the classic use case.

> [!note] Two auth models
> This note covers **JWT (stateless)** auth. Colt Steele's bootcamp teaches **session-based (stateful)** auth with `express-session` + Passport + `connect-mongo`. Both are valid — see [Session vs JWT](#session-vs-jwt-which-to-use) at the bottom for when to pick which.

---

## 0. Issuing the Token (the missing first step)

Verification is pointless without issuance. You need a login route that checks credentials and hands back a signed token.

```javascript
// routes/auth.js
const jwt = require('jsonwebtoken');
const bcrypt = require('bcrypt');
const User = require('../models/User');

router.post('/login', async (req, res) => {
    const { email, password } = req.body;

    const user = await User.findOne({ email });
    if (!user) {
        // Same message for both failure modes — don't leak which emails exist
        return res.status(401).json({ error: 'Invalid credentials.' });
    }

    const match = await bcrypt.compare(password, user.passwordHash);
    if (!match) {
        return res.status(401).json({ error: 'Invalid credentials.' });
    }

    const token = jwt.sign(
        { id: user._id, role: user.role },   // payload — keep it minimal
        process.env.JWT_SECRET,
        { expiresIn: '15m' }
    );

    res.json({ token });
});
```

> [!warning] JWT payloads are not encrypted
> A JWT is base64-**encoded**, not encrypted. Anyone holding the token can decode and read the payload at [jwt.io](https://jwt.io). The signature only guarantees the payload hasn't been *tampered with* — it does not hide it. Never put passwords, hashes, PII, or secrets in there. An id and a role are enough.

---

## 1. Create the Authentication Middleware

Create a file named `authMiddleware.js`. This function extracts the token from the incoming request headers, verifies it, and attaches the authenticated user data to the request object.

```javascript
// authMiddleware.js
const jwt = require('jsonwebtoken');

const protectRoute = (req, res, next) => {
    // 1. Get the token from the Authorization header
    const authHeader = req.headers.authorization;

    if (!authHeader || !authHeader.startsWith('Bearer ')) {
        return res.status(401).json({ error: 'Access denied. No token provided.' });
    }

    const token = authHeader.split(' ')[1];

    try {
        // 2. Verify the token using your JWT secret
        const decoded = jwt.verify(token, process.env.JWT_SECRET);

        // 3. Attach user payload to the request object
        req.user = decoded;

        // 4. Pass control to the next middleware or route handler
        next();
    } catch (error) {
        res.status(401).json({ error: 'Invalid or expired token.' });
    }
};

module.exports = protectRoute;
```

### Distinguishing failure reasons

The catch-all above tells the client nothing useful. `jsonwebtoken` throws typed errors, and the expired case matters because that's the client's cue to hit the refresh endpoint rather than send the user back to login.

```javascript
} catch (error) {
    if (error.name === 'TokenExpiredError') {
        return res.status(401).json({
            error: 'Token expired.',
            code: 'TOKEN_EXPIRED'          // client uses this to trigger refresh
        });
    }
    if (error.name === 'JsonWebTokenError') {
        return res.status(401).json({ error: 'Malformed token.' });
    }
    return res.status(401).json({ error: 'Authentication failed.' });
}
```

### Trusting the payload vs. re-fetching the user

`req.user = decoded` is fast — zero DB calls — but the payload is a **snapshot from issuance time**. If you demote an admin or ban an account, their existing token still says `role: 'admin'` until it expires.

```javascript
// Safer variant: verify the user still exists and is active
const decoded = jwt.verify(token, process.env.JWT_SECRET);
const user = await User.findById(decoded.id).select('-passwordHash');

if (!user || user.isBanned) {
    return res.status(401).json({ error: 'Account no longer valid.' });
}

req.user = user;
next();
```

Trade-off: one DB round-trip per protected request. With short token lifetimes (15 min) the stale-payload window is small, so trusting the payload is usually fine. Re-fetch on the routes where a stale role would actually cause damage.

---

## 2. Apply Protection to Specific Routes

Three ways to wire it up, depending on your architecture.

### Method A: Protect Individual Routes

Pass the middleware as the second argument in your route definition. Express executes callbacks sequentially, left to right.

```javascript
const express = require('express');
const protectRoute = require('./authMiddleware');
const app = express();

// Public route
app.get('/api/products', (req, res) => {
    res.json({ message: 'Public product listings' });
});

// Protected route
app.get('/api/dashboard', protectRoute, (req, res) => {
    // Access the attached user data from the middleware
    res.json({ message: `Welcome to your dashboard, user ${req.user.id}` });
});
```

### Method B: Protect an Entire Router Instance

If you group related endpoints using `express.Router()`, apply the middleware to that router via `router.use()`.

```javascript
const express = require('express');
const protectRoute = require('./authMiddleware');
const router = express.Router();

// Apply protection to EVERY route inside this router
router.use(protectRoute);

// Both endpoints below are automatically protected
router.get('/profile', (req, res) => res.json(req.user));
router.put('/settings', (req, res) => res.json({ status: 'updated' }));

module.exports = router;
```

### Method B2: Mount-Path Protection

You can also gate an entire URL prefix from `app.js`, without touching the router file at all:

```javascript
// Everything under /api/admin requires a valid token
app.use('/api/admin', protectRoute, adminRouter);
```

Useful when the router is third-party or shared, and you want the auth decision to live in one place.

### Method C: Multi-Layered Protection (Role-Based Access)

Stack multiple middleware functions to enforce deeper permissions — validating administrative privileges *after* validating authentication.

```javascript
const checkAdmin = (req, res, next) => {
    if (req.user && req.user.role === 'admin') {
        next();
    } else {
        res.status(403).json({ error: 'Forbidden. Admin rights required.' });
    }
};

// Requires both valid login AND admin role
app.delete('/api/users/:id', protectRoute, checkAdmin, (req, res) => {
    res.json({ message: 'User successfully deleted.' });
});
```

### Method D: Ownership Checks (the one people forget)

Role checks answer *"what kind of user are you?"* They don't answer *"is this thing yours?"* Without an ownership check, any logged-in user can edit any other user's records just by changing the id in the URL — one of the most common real-world API vulnerabilities (IDOR: Insecure Direct Object Reference).

```javascript
const isOwner = async (req, res, next) => {
    const post = await Post.findById(req.params.id);

    if (!post) {
        return res.status(404).json({ error: 'Not found.' });
    }
    if (!post.author.equals(req.user.id)) {
        return res.status(403).json({ error: 'You do not own this resource.' });
    }

    req.post = post;   // avoid re-querying in the handler
    next();
};

app.put('/api/posts/:id', protectRoute, isOwner, updatePost);
```

### Reusable Role Factory

Rather than writing `checkAdmin`, `checkEditor`, `checkModerator` separately, return middleware from a function:

```javascript
const requireRole = (...allowedRoles) => {
    return (req, res, next) => {
        if (!req.user || !allowedRoles.includes(req.user.role)) {
            return res.status(403).json({ error: 'Insufficient permissions.' });
        }
        next();
    };
};

// Usage
app.delete('/api/users/:id', protectRoute, requireRole('admin'), deleteUser);
app.post('/api/articles', protectRoute, requireRole('admin', 'editor'), createArticle);
```

---

## 3. Async Middleware and Error Handling

If your middleware is `async`, a thrown error will **not** be caught by Express's error handler automatically (in Express 4). It becomes an unhandled promise rejection and the request hangs.

```javascript
// Wrapper — catches async rejections and forwards to Express
const catchAsync = (fn) => (req, res, next) => {
    Promise.resolve(fn(req, res, next)).catch(next);
};

const protectRoute = catchAsync(async (req, res, next) => {
    const user = await User.findById(decoded.id);
    // ...
    next();
});
```

Then define a global error handler **last**, after all routes:

```javascript
// Must have exactly 4 params for Express to recognise it as an error handler
app.use((err, req, res, next) => {
    const status = err.status || 500;
    res.status(status).json({
        error: err.message || 'Internal server error'
    });
});
```

> Express 5 handles async errors natively, making `catchAsync` unnecessary. Check your version before adding it.

---

## 4. Cookie-Based Tokens (`HttpOnly`)

Storing a JWT in `localStorage` leaves it readable by any JavaScript on the page, so a single XSS bug leaks the token. `HttpOnly` cookies are invisible to JS.

```javascript
const cookieParser = require('cookie-parser');
app.use(cookieParser());

// On login
res.cookie('token', token, {
    httpOnly: true,                                  // JS cannot read it
    secure: process.env.NODE_ENV === 'production',   // HTTPS only in prod
    sameSite: 'strict',                              // blocks most CSRF
    maxAge: 15 * 60 * 1000
});
```

Then read it in the middleware, falling back to the header:

```javascript
const token = req.cookies.token
    || (req.headers.authorization?.startsWith('Bearer ')
        ? req.headers.authorization.split(' ')[1]
        : null);
```

> [!warning] Cookies introduce CSRF risk
> Because browsers attach cookies automatically, a malicious site can trigger authenticated requests on the user's behalf. `sameSite: 'strict'` mitigates most of this. For cross-origin setups you'll need `sameSite: 'none'` plus a CSRF token. Header-based tokens don't have this problem — the trade is XSS risk against CSRF risk.

---

## 5. Refresh Tokens & Logout

Short-lived access tokens are good security but bad UX — users get logged out every 15 minutes. The standard fix is a token pair:

| Token | Lifetime | Stored | Purpose |
|---|---|---|---|
| Access | 15 min | Memory / `HttpOnly` cookie | Sent with every request |
| Refresh | 7–30 days | `HttpOnly` cookie **and** DB | Mints new access tokens |

```javascript
router.post('/refresh', async (req, res) => {
    const { refreshToken } = req.cookies;
    if (!refreshToken) return res.status(401).json({ error: 'No refresh token.' });

    // Must exist in DB — this is what makes revocation possible
    const stored = await RefreshToken.findOne({ token: refreshToken });
    if (!stored) return res.status(401).json({ error: 'Invalid refresh token.' });

    const decoded = jwt.verify(refreshToken, process.env.REFRESH_SECRET);
    const accessToken = jwt.sign(
        { id: decoded.id, role: decoded.role },
        process.env.JWT_SECRET,
        { expiresIn: '15m' }
    );

    res.json({ accessToken });
});
```

**The logout problem:** a signed JWT stays valid until it expires. You cannot "delete" it server-side. Real logout means deleting the refresh token from the DB (so no new access tokens can be minted) and clearing the cookie. The current access token stays technically valid for its remaining minutes — which is exactly why you keep that window short.

---

## Best Practices Checklist

- [ ] **Order matters.** Define public endpoints (`/login`, `/signup`) *above* any catch-all protection middleware, or you'll lock users out of the very pages they need to authenticate on.
- [ ] **Secure environment variables.** Keep secrets in a `.env` file via [dotenv](https://www.npmjs.com/package/dotenv); never hardcode them. Add `.env` to `.gitignore` before your first commit, not after.
- [ ] **Use a strong secret.** `openssl rand -base64 64`. Not `"secret"`, not your project name.
- [ ] **Always set `expiresIn`.** A token without expiry is a permanent credential.
- [ ] **401 vs 403.** `401 Unauthorized` = "I don't know who you are" (bad/missing token). `403 Forbidden` = "I know who you are, and you're not allowed" (valid token, insufficient role). Getting these right makes client-side handling much cleaner.
- [ ] **Hash passwords with bcrypt or argon2.** Never store plaintext, never use MD5/SHA for passwords.
- [ ] **Rate-limit the login route.** [express-rate-limit](https://www.npmjs.com/package/express-rate-limit) — otherwise credential stuffing is trivial.
- [ ] **Add [helmet](https://www.npmjs.com/package/helmet).** One line, sets sensible security headers.
- [ ] **Validate input.** Use `zod` or `joi` on request bodies. An auth route is the first thing attackers probe.
- [ ] **Alternative storage.** [cookie-parser](https://www.npmjs.com/package/cookie-parser) extracts tokens from `HttpOnly` cookies if you'd rather avoid headers.

---

## Session vs JWT: Which to Use?

| | Session (stateful) | JWT (stateless) |
|---|---|---|
| **Where state lives** | Server (Mongo/Redis store) | In the token itself |
| **Revocation** | Instant — delete the session | Hard — must wait for expiry |
| **Scaling** | Needs shared session store | Any server can verify |
| **Best for** | Traditional server-rendered apps | APIs, mobile clients, microservices |
| **Complexity** | Lower | Higher (refresh flow, storage decisions) |

For a monolithic Express + EJS app — the bootcamp shape — sessions with Passport are simpler and give you real logout for free. Reach for JWT when you have a separate frontend, a mobile client, or services that need to verify identity without a shared session store.

---

## Testing It Quickly

```bash
# 1. Get a token
curl -X POST http://localhost:3000/api/login \
  -H "Content-Type: application/json" \
  -d '{"email":"me@example.com","password":"hunter2"}'

# 2. Hit a protected route without one → expect 401
curl http://localhost:3000/api/dashboard

# 3. Hit it with the token → expect 200
curl http://localhost:3000/api/dashboard \
  -H "Authorization: Bearer <paste_token_here>"
```

---

## Related Notes

- [[Express Middleware Basics]]
- [[Mongoose Models and Schemas]]
- [[bcrypt Password Hashing]]
- [[Error Handling in Express]]